In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# ------------- config -------------
ROUND0_DIR = Path(".")  # run from data/round0
METHODS = [
    ("round0_additive_panel.csv",  "additive"),
    ("round0_plldelta_panel.csv",  "plldelta"),
    ("round0_mutant_ctx_panel.csv","mutant_ctx"),
]

# DMS candidates (first one that exists will be used)
DMS_CANDIDATES = [
    Path("zeroshot_prior/gfp_dms.csv"),
    Path("../zeroshot_prior/gfp_dms.csv"),
    Path("../../zeroshot_prior/gfp_dms.csv"),
    Path("gfp_dms.csv"),
    Path("../gfp_dms.csv"),
    Path("../../gfp_dms.csv"),
]

# possible measurement column names (first match wins)
MEASURED_COL_CANDIDATES = [
    "measured_score", "fitness", "score", "fluorescence",
    "log_fitness", "log2_enrichment", "enrichment" , "DMS_score"
]

# ------------- helpers -------------
def find_dms():
    for p in DMS_CANDIDATES:
        if p.exists():
            return p
    raise FileNotFoundError(
        f"Could not find DMS file. Looked for: {', '.join(map(str, DMS_CANDIDATES))}"
    )

def pick_measured_col(df: pd.DataFrame) -> str:
    for c in MEASURED_COL_CANDIDATES:
        if c in df.columns:
            return c
    raise KeyError(
        f"None of {MEASURED_COL_CANDIDATES} found in DMS columns: {list(df.columns)}"
    )

def ensure_panel_schema(df: pd.DataFrame, tag: str) -> pd.DataFrame:
    need = ["mutated_sequence","num_subs","score","method","is_WT","is_blank"]
    missing = [c for c in need if c not in df.columns]
    if missing:
        # allow older panels (without flags); add defaults
        if "is_WT" not in df.columns: df["is_WT"] = (df["mutated_sequence"]=="WT").astype(int)
        if "is_blank" not in df.columns: df["is_blank"] = (df["mutated_sequence"]=="BLANK").astype(int)
        still_missing = [c for c in need if c not in df.columns]
        if still_missing:
            raise ValueError(f"[{tag}] panel missing columns {still_missing}")
    # strong types
    df = df.copy()
    df["mutated_sequence"] = df["mutated_sequence"].astype(str)
    df["num_subs"] = pd.to_numeric(df["num_subs"], errors="coerce")
    df["score"]    = pd.to_numeric(df["score"], errors="coerce")
    df["method"]   = df["method"].astype(str)
    df["is_WT"]    = df["is_WT"].astype(int)
    df["is_blank"] = df["is_blank"].astype(int)
    return df

# ------------- main -------------
if __name__ == "__main__":
    dms_path = find_dms()
    dms = pd.read_csv(dms_path)
    # normalize DMS keys/types
    if "mutated_sequence" not in dms.columns:
        raise KeyError(f"'mutated_sequence' column not found in DMS: {dms_path}")
    dms["mutated_sequence"] = dms["mutated_sequence"].astype(str)

    meas_col = pick_measured_col(dms)
    dms[meas_col] = pd.to_numeric(dms[meas_col], errors="coerce")

    # WT baseline from DMS: prefer rows with num_subs==0, else any exact WT record if present
    if "num_subs" in dms.columns:
        wt_rows = dms[dms["num_subs"]==0]
    else:
        wt_rows = dms[dms["mutated_sequence"].str.upper().eq("WT")]
    WT_value = np.nan
    if len(wt_rows):
        WT_value = float(wt_rows[meas_col].median())

    print(f"Using DMS: {dms_path} | measured column: {meas_col} | WT baseline: {WT_value}")

    for panel_file, tag in METHODS:
        p = ROUND0_DIR / panel_file
        if not p.exists():
            print(f"[{tag}] {panel_file} not found, skipping.")
            continue

        panel = ensure_panel_schema(pd.read_csv(p), tag)

        # Merge measured score from DMS onto PANEL rows (per-well)
        lab = panel.merge(
            dms[["mutated_sequence", meas_col]],
            on="mutated_sequence", how="left"
        ).rename(columns={meas_col: "measured_score"})

        # Handle controls
        # - BLANK: NaN
        # - WT: fill from DMS baseline if missing
        is_blank = lab["is_blank"] == 1
        lab.loc[is_blank, "measured_score"] = np.nan

        is_wt = lab["is_WT"] == 1
        if not np.isnan(WT_value):
            lab.loc[ is_wt & lab["measured_score"].isna(), "measured_score"] = WT_value

        # n_reps and sd per sequence (computed across duplicate wells)
        grp = lab.groupby("mutated_sequence")["measured_score"]
        lab["n_reps"] = grp.transform("count")
        lab["sd"]     = grp.transform("std").fillna(0.0)

        # Reorder/clean columns
        keep = ["mutated_sequence","num_subs","score","method",
                "is_WT","is_blank","measured_score","n_reps","sd"]
        if "well" in lab.columns:
            keep = ["well"] + keep
        lab = lab[keep]

        out = ROUND0_DIR / f"round0_{tag}_labeled.csv"
        lab.to_csv(out, index=False)

        # Quick summary
        matched = lab["measured_score"].notna().sum()
        total   = len(lab)
        uniq    = lab.drop_duplicates("mutated_sequence")
        print(f"[{tag}] wrote {out.name} | matched: {matched}/{total} rows | unique seqs: {len(uniq)}")


Using DMS: ..\zeroshot_prior\gfp_dms.csv | measured column: DMS_score | WT baseline: nan
[additive] wrote round0_additive_labeled.csv | matched: 88/96 rows | unique seqs: 86
[plldelta] wrote round0_plldelta_labeled.csv | matched: 88/96 rows | unique seqs: 86
[mutant_ctx] wrote round0_mutant_ctx_labeled.csv | matched: 88/96 rows | unique seqs: 86
